# Dunnhumby 최종 test 10시드 — 가치기저 M2
교수님 지시 프로토콜입니다. 검증 분할과 선택 절차 없이 train+val을 합쳐 100 epoch 고정 학습하고, 보호된 test 분할을 arm당 정확히 1회 평가한 뒤 10개 시드(42~51)의 평균과 95% 신뢰구간을 보고합니다.

arm 3개: **M1**(원 LightGCN, K=1) / **M2**(q_C 배율 가치기저) / **M2-CLV순열대조**(degree 10분위 안에서 q_N·q_V·q_C·valid를 함께 섞음).

M4는 재설계 예정이므로 포함하지 않습니다. 나중에 M4 arm을 추가해도 완료된 M1·M2 시드는 캐시에서 재사용됩니다.

**총 30회 학습(약 20시간)**입니다. 세션이 끊기면 같은 노트북을 다시 실행하세요 — epoch 단위로 자동 재개하고, 완료된 arm은 test를 다시 평가하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'PLACEHOLDER_SHA'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_value_basis_test10 as final

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert final.CODE_VERSION == 'm2-value-basis-test10-v1'
cfg = final.configure_value_basis_test10()
summary = final.preflight_summary(cfg)
assert summary['seeds'] == list(range(42, 52))
assert summary['validation_selection'] is False
assert summary['test_evaluations_per_arm'] == 1
assert summary['loss']['negative_count'] == 1
assert summary['loss']['row_weighting'] is False
assert summary['m2']['rho'] == 0.25
assert summary['m4_present'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = final.run_value_basis_test10(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) 10시드 절대지표 평균과 95% 신뢰구간')
show(result_df.attrs['absolute_summary'])
print('2) 동일 seed 차이의 10시드 평균과 95% 신뢰구간')
show(result_df.attrs['paired_summary'])
print('3) 시드별 절대지표')
show(result_df)
print('4) 사전 판정')
print(json.dumps(result_df.attrs['reading'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
